# Light or Letdown? — Social Media Analysis of the **Ferrari Luce**
### Web and Social Media Search and Analysis — BSc AI, UniMiB

Self-contained Colab notebook for the Track-1 project (Network + Content + Visualization).
It reproduces the tested project code (the `src/` package) inside Colab and runs the full pipeline.

**How to use**
1. *Runtime → Change runtime type → GPU* (recommended; speeds up the transformer models).
2. Run the cells top to bottom.
3. Enter your **Bluesky App Password** and **Reddit** (read-only) credentials when prompted — they are kept only in memory, never written to disk.

**Pipeline:** collect (LAB 1–2) → graph + centralities (LAB 3) → communities (LAB 4) → sentiment/emotion/ABSA (LAB 5) → sarcasm/stance/topics/language → NER (LAB 6) → visualization.
See `MEGAPLAN.md` in the repo for the full design and research questions.


## 1. Install dependencies
Colab already bundles torch, transformers, pandas, numpy, scikit-learn, networkx, nltk, matplotlib, seaborn, wordcloud and spaCy. We install the rest.

In [ ]:
!pip install -q atproto praw vaderSentiment afinn NRCLex langdetect pyvis python-louvain python-dotenv bertopic
!python -m spacy download en_core_web_sm -q
print("installed.")

In [ ]:
import nltk
for pkg in ("punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"):
    nltk.download(pkg, quiet=True)
print("NLTK data ready.")

## 2. Credentials (entered securely, not stored)
- **Bluesky:** create an *App Password* at bsky.app → Settings → App Passwords.
- **Reddit:** register a **script** app at https://www.reddit.com/prefs/apps — read-only mode needs only client id + secret.

> Tip: you can instead use Colab **Secrets** (the 🔑 panel) and read them with `google.colab.userdata.get(...)`.

In [ ]:
import os, getpass
os.environ["BLUESKY_HANDLE"] = input("Bluesky handle (e.g. you.bsky.social): ").strip()
os.environ["BLUESKY_APP_PASSWORD"] = getpass.getpass("Bluesky App Password: ").strip()
os.environ["REDDIT_CLIENT_ID"] = getpass.getpass("Reddit client_id: ").strip()
os.environ["REDDIT_CLIENT_SECRET"] = getpass.getpass("Reddit client_secret: ").strip()
_u = input("Reddit username (for the user-agent string): ").strip() or "student"
os.environ["REDDIT_USER_AGENT"] = f"wsa-ferrari-luce/0.1 by u/{_u}"
print("Credentials set for this session only.")

## 3. Project code
Write the tested `src/` package into the Colab filesystem (single source of truth).

In [ ]:
import os
for d in ("src", "data/raw", "data/processed", "figures", "report"):
    os.makedirs(d, exist_ok=True)
print("folders ready.")

In [ ]:
%%writefile src/__init__.py
"""WSA Ferrari Luce project package."""

In [ ]:
%%writefile src/config.py
"""Central configuration: paths, queries, aspect lexicons, models, event dates.

Everything tunable lives here so the notebooks/scripts stay thin.
"""
from __future__ import annotations
from pathlib import Path

# ----------------------------------------------------------------------------
# Paths
# ----------------------------------------------------------------------------
BASE_DIR = Path(__file__).resolve().parent.parent
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"
FIGURES = BASE_DIR / "figures"

for _p in (DATA_RAW, DATA_PROCESSED, FIGURES):
    _p.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------------------
# Topic definition: queries / subreddits / accounts
# ----------------------------------------------------------------------------
# Bluesky search supports Boolean ops:  space=AND, "|"=OR, "-"=NOT, quotes=phrase
BLUESKY_QUERIES = [
    '"Ferrari Luce"',
    "Ferrari Luce",
    '"Ferrari EV"',
    "Ferrari electric",
    "Ferrari Elettrica",
    "#FerrariLuce",
    "Ferrari Luce ugly",          # surface the backlash explicitly
]

REDDIT_QUERIES = [
    "Ferrari Luce",
    "Ferrari EV",
    "Ferrari electric",
    "Ferrari Elettrica",
    '"Luce"',
]

SUBREDDITS = [
    "Ferrari",
    "cars",
    "electricvehicles",
    "formula1",
    "stocks",
    "wallstreetbets",
]

# Bluesky handles whose ego-network (followers/follows) we optionally crawl.
# Fill in real handles you discover during collection, e.g. "ferrari.com".
KEY_BLUESKY_ACCOUNTS: list[str] = []

# ----------------------------------------------------------------------------
# Time window (the Luce story)
# ----------------------------------------------------------------------------
# Capital Markets Day "Elettrica" preview -> production reveal -> stock drop.
SINCE_ISO = "2025-10-01T00:00:00Z"     # widest window (since CMD)
FOCUS_SINCE_ISO = "2026-05-20T00:00:00Z"  # the dense reveal window
UNTIL_ISO = None                        # None = up to now
REDDIT_TIME_FILTER = "year"             # PRAW search: all/year/month/week/day

# Event markers for the temporal-sentiment chart (annotated manually, not scraped)
EVENT_DATES = [
    ("2025-10-09", "Capital Markets Day (Elettrica preview)"),
    ("2026-05-25", "Luce reveal near Rome"),
    ("2026-05-26", "Stock falls ~8%"),
]

# ----------------------------------------------------------------------------
# Aspect lexicons for Aspect-Based Sentiment Analysis (LAB 5, #CrazyPizza style)
# Each post is tagged with every aspect whose keywords it matches.
# ----------------------------------------------------------------------------
ASPECTS: dict[str, list[str]] = {
    "exterior_look": [
        "design", "look", "looks", "styling", "ugly", "beautiful", "shape",
        "proportions", "silhouette", "aesthetic", "hideous", "gorgeous",
        "bubble", "blob", "egg", "front", "rear",
    ],
    "interior": [
        "interior", "cabin", "seats", "seat", "five-seater", "5-seater",
        "five seater", "practical", "space", "dashboard", "screen", "back seat",
    ],
    "sound": [
        "sound", "noise", "engine note", "exhaust", "silent", "vibration",
        "artificial sound", "fake sound", "soundtrack", "roar", "audio",
    ],
    "price": [
        "price", "expensive", "cost", "euro", "550", "640", "money", "value",
        "overpriced", "cheap", "worth", "afford", "$",
    ],
    "performance": [
        "performance", "power", "fast", "acceleration", "0-100", "0 to 60",
        "top speed", "range", "horsepower", "battery", "torque", "quick", "cv",
    ],
    "name": ["name", "luce", "called", "naming", "named"],
    "electrification": [
        "electric", "ev", "battery", "charging", "charge", "combustion",
        "ice", "petrol", "gas", "hybrid", "electrify", "electrification", "evs",
    ],
    "brand_identity": [
        "heritage", "soul", "real ferrari", "not a ferrari", "tradition",
        "prancing horse", "brand", "identity", "betrayal", "maranello",
        "sellout", "legacy",
    ],
    "design_house": [
        "jony ive", "ive", "apple", "lovefrom", "designer", "iphone", "tim cook",
    ],
}

# Comparison entities we expect to recur (used to sanity-check NER output)
WATCH_ENTITIES = [
    "Nissan Leaf", "Tesla", "Apple", "Jony Ive", "LoveFrom", "Benedetto Vigna",
    "NIO", "Maranello", "Ferrari",
]

# Languages of analytical interest (RQ8: Italian national-pride angle)
LANGUAGES = ["en", "it"]

# ----------------------------------------------------------------------------
# Models
# ----------------------------------------------------------------------------
SENTIMENT_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
IRONY_MODEL = "cardiffnlp/twitter-roberta-base-irony"
OFFENSIVE_MODEL = "cardiffnlp/twitter-roberta-base-offensive"
STANCE_NLI_MODEL = "facebook/bart-large-mnli"
SPACY_MODEL = "en_core_web_sm"

# Stance targets (zero-shot NLI). Each maps to a hypothesis template.
STANCE_TARGETS = {
    "luce": "This text is in favor of the Ferrari Luce car.",
    "ev_transition": "This text is in favor of electric vehicles.",
}

# ----------------------------------------------------------------------------
# Collection caps (be polite to the APIs)
# ----------------------------------------------------------------------------
BLUESKY_MAX_PER_QUERY = 1000
BLUESKY_PAGE_SIZE = 25          # API hard max for search_posts
REDDIT_MAX_PER_QUERY = 500
PULLPUSH_PAGE_SIZE = 100

In [ ]:
%%writefile src/utils.py
"""Shared helpers: credentials, retry/backoff, text cleaning, I/O, language id.

Heavy / optional dependencies (nltk, langdetect) are imported lazily so that
collectors don't need the full ML stack, and so a missing package only breaks
the feature that uses it.
"""
from __future__ import annotations
import os
import re
import time
import logging
from pathlib import Path
from typing import Callable, Iterable

import pandas as pd

from . import config

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("wsa")

# ----------------------------------------------------------------------------
# Credentials
# ----------------------------------------------------------------------------
def load_env() -> None:
    """Load .env into os.environ (no-op if python-dotenv is missing)."""
    try:
        from dotenv import load_dotenv
        load_dotenv(config.BASE_DIR / ".env")
    except Exception:
        log.warning("python-dotenv not available; relying on existing env vars.")


def require_env(*keys: str) -> dict[str, str]:
    load_env()
    missing = [k for k in keys if not os.environ.get(k)]
    if missing:
        raise RuntimeError(
            f"Missing credentials in environment/.env: {missing}. "
            f"Copy .env.example to .env and fill them in."
        )
    return {k: os.environ[k] for k in keys}


# ----------------------------------------------------------------------------
# Retry / backoff (LAB 2 pattern)
# ----------------------------------------------------------------------------
def with_retries(
    func: Callable,
    *args,
    max_attempts: int = 5,
    base: float = 2.0,
    max_delay: float = 60.0,
    exceptions: tuple = (Exception,),
    **kwargs,
):
    """Call func with exponential backoff on transient errors / rate limits."""
    for attempt in range(max_attempts):
        try:
            return func(*args, **kwargs)
        except exceptions as e:  # noqa: BLE001 - intentionally broad, re-raised below
            if attempt == max_attempts - 1:
                raise
            delay = min(base ** attempt, max_delay)
            log.warning("attempt %d/%d failed (%s); sleeping %.1fs",
                        attempt + 1, max_attempts, type(e).__name__, delay)
            time.sleep(delay)


# ----------------------------------------------------------------------------
# Text cleaning  (LAB 1 regex skills reused here instead of scraping news)
# ----------------------------------------------------------------------------
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@[\w.]+")
HASHTAG_RE = re.compile(r"#(\w+)")
WS_RE = re.compile(r"\s+")


def clean_basic(text: str) -> str:
    """Light clean: strip URLs, collapse whitespace. Keeps #/@ as signal."""
    if not isinstance(text, str):
        return ""
    text = URL_RE.sub(" ", text)
    return WS_RE.sub(" ", text).strip()


def clean_for_transformer(text: str) -> str:
    """cardiffnlp-recommended normalisation: @user / http placeholders.

    Used for the BPE / transformer lane (Lane B). Do NOT stopword/lemmatise here.
    """
    if not isinstance(text, str):
        return ""
    out = []
    for tok in text.split():
        if tok.startswith("@") and len(tok) > 1:
            out.append("@user")
        elif tok.startswith("http") or tok.startswith("www."):
            out.append("http")
        else:
            out.append(tok)
    return WS_RE.sub(" ", " ".join(out)).strip()


_NLTK_READY = False


def _ensure_nltk() -> None:
    global _NLTK_READY
    if _NLTK_READY:
        return
    import nltk
    for pkg in ("punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"):
        try:
            nltk.download(pkg, quiet=True)
        except Exception:
            pass
    _NLTK_READY = True


def tokens_for_lexicon(text: str, lang: str = "english") -> list[str]:
    """Word-level lane (Lane A): TweetTokenizer + lowercase + stopwords + lemma.

    Feeds VADER/AFINN/NRC word stats, TF-IDF and word clouds.
    """
    if not isinstance(text, str) or not text.strip():
        return []
    _ensure_nltk()
    from nltk.tokenize import TweetTokenizer
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer

    tok = TweetTokenizer(preserve_case=False, reduce_len=True, strip_handles=False)
    lemm = WordNetLemmatizer()
    try:
        stop = set(stopwords.words(lang))
    except Exception:
        stop = set()
    out = []
    for t in tok.tokenize(clean_basic(text)):
        if t in stop:
            continue
        if not re.search(r"[a-zA-Z#]", t):  # drop pure punctuation/numbers
            continue
        out.append(lemm.lemmatize(t))
    return out


def extract_hashtags(text: str) -> list[str]:
    return [h.lower() for h in HASHTAG_RE.findall(text or "")]


# ----------------------------------------------------------------------------
# Language detection (RQ8)
# ----------------------------------------------------------------------------
def detect_lang(text: str) -> str:
    if not isinstance(text, str) or len(text.strip()) < 3:
        return "unknown"
    try:
        from langdetect import detect, DetectorFactory
        DetectorFactory.seed = 0
        return detect(text)
    except Exception:
        return "unknown"


# ----------------------------------------------------------------------------
# I/O helpers
# ----------------------------------------------------------------------------
def save_csv(df: pd.DataFrame, name: str, processed: bool = True) -> Path:
    folder = config.DATA_PROCESSED if processed else config.DATA_RAW
    path = folder / name
    df.to_csv(path, index=False)
    log.info("saved %d rows -> %s", len(df), path)
    return path


def load_csv(name: str, processed: bool = True) -> pd.DataFrame:
    folder = config.DATA_PROCESSED if processed else config.DATA_RAW
    return pd.read_csv(folder / name)


def dedup(records: Iterable[dict], key: str) -> list[dict]:
    seen, out = set(), []
    for r in records:
        k = r.get(key)
        if k in seen:
            continue
        seen.add(k)
        out.append(r)
    return out

In [ ]:
%%writefile src/collect_bluesky.py
"""LAB 2 — Bluesky (AT Protocol) collector.

Collects posts matching the topic queries (with engagement + facets + thread
links) and optionally the follow graph of key accounts.

Run:  python -m src.collect_bluesky
Outputs:  data/processed/posts_bluesky.csv , data/processed/edges_follows.csv
"""
from __future__ import annotations
import time
import pandas as pd

from . import config
from .utils import log, require_env, with_retries, save_csv, dedup


def get_client():
    """Authenticate with a Bluesky App Password (never the main password)."""
    creds = require_env("BLUESKY_HANDLE", "BLUESKY_APP_PASSWORD")
    from atproto import Client
    client = Client()
    with_retries(client.login, creds["BLUESKY_HANDLE"], creds["BLUESKY_APP_PASSWORD"])
    log.info("Bluesky login OK as %s", creds["BLUESKY_HANDLE"])
    return client


def _facets(record) -> tuple[list[str], list[str], list[str]]:
    """Extract (hashtags, mention DIDs, links) from a post record's facets."""
    tags, mentions, links = [], [], []
    for facet in (getattr(record, "facets", None) or []):
        for feat in (getattr(facet, "features", None) or []):
            tag = getattr(feat, "tag", None)
            did = getattr(feat, "did", None)
            uri = getattr(feat, "uri", None)
            if tag:
                tags.append(tag.lower())
            if did:
                mentions.append(did)
            if uri:
                links.append(uri)
    return tags, mentions, links


def _flatten(post) -> dict:
    record = getattr(post, "record", None)
    author = getattr(post, "author", None)
    tags, mentions, links = _facets(record)
    reply = getattr(record, "reply", None)
    return {
        "uri": getattr(post, "uri", None),
        "cid": getattr(post, "cid", None),
        "created_at": getattr(record, "created_at", None),
        "text": getattr(record, "text", "") or "",
        "author_handle": getattr(author, "handle", None),
        "author_did": getattr(author, "did", None),
        "author_display_name": getattr(author, "display_name", None),
        "like_count": getattr(post, "like_count", 0) or 0,
        "repost_count": getattr(post, "repost_count", 0) or 0,
        "reply_count": getattr(post, "reply_count", 0) or 0,
        "quote_count": getattr(post, "quote_count", 0) or 0,
        "langs": ",".join(getattr(record, "langs", None) or []),
        "hashtags": ",".join(tags),
        "mention_dids": ",".join(mentions),
        "links": ",".join(links),
        "reply_parent_uri": getattr(getattr(reply, "parent", None), "uri", None) if reply else None,
        "reply_root_uri": getattr(getattr(reply, "root", None), "uri", None) if reply else None,
    }


def search_query(client, query: str, max_results: int = config.BLUESKY_MAX_PER_QUERY,
                 since: str | None = config.SINCE_ISO, until: str | None = config.UNTIL_ISO) -> list[dict]:
    """Cursor-paginated search_posts for a single query (LAB 2)."""
    out: list[dict] = []
    cursor = None
    while len(out) < max_results:
        params = {"q": query, "limit": config.BLUESKY_PAGE_SIZE, "sort": "latest"}
        if cursor:
            params["cursor"] = cursor
        if since:
            params["since"] = since
        if until:
            params["until"] = until
        resp = with_retries(client.app.bsky.feed.search_posts, params)
        posts = getattr(resp, "posts", None) or []
        if not posts:
            break
        out.extend(_flatten(p) for p in posts)
        cursor = getattr(resp, "cursor", None)
        if not cursor:
            break
        time.sleep(0.4)  # be polite
    log.info("  query %-28s -> %d posts", query, len(out))
    return out[:max_results]


def collect_posts(queries: list[str] | None = None) -> pd.DataFrame:
    client = get_client()
    queries = queries or config.BLUESKY_QUERIES
    rows: list[dict] = []
    for q in queries:
        rows.extend(search_query(client, q))
    rows = dedup(rows, key="uri")
    df = pd.DataFrame(rows)
    if not df.empty:
        df["source"] = "bluesky"
        df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
    save_csv(df, "posts_bluesky.csv")
    return df


def collect_follow_edges(accounts: list[str] | None = None) -> pd.DataFrame:
    """Optional follower/follows ego-networks for key accounts."""
    accounts = accounts or config.KEY_BLUESKY_ACCOUNTS
    if not accounts:
        log.info("No KEY_BLUESKY_ACCOUNTS configured; skipping follow graph.")
        return pd.DataFrame(columns=["source", "target", "relation"])
    client = get_client()
    edges: list[dict] = []
    for actor in accounts:
        for relation, endpoint, field in (
            ("follows", client.app.bsky.graph.get_follows, "follows"),
            ("followed_by", client.app.bsky.graph.get_followers, "followers"),
        ):
            cursor = None
            while True:
                params = {"actor": actor, "limit": 100}
                if cursor:
                    params["cursor"] = cursor
                resp = with_retries(endpoint, params)
                people = getattr(resp, field, None) or []
                for p in people:
                    other = getattr(p, "handle", None)
                    if relation == "follows":
                        edges.append({"source": actor, "target": other, "relation": "follows"})
                    else:
                        edges.append({"source": other, "target": actor, "relation": "follows"})
                cursor = getattr(resp, "cursor", None)
                if not cursor:
                    break
    df = pd.DataFrame(edges)
    save_csv(df, "edges_follows.csv")
    return df


def main() -> None:
    df = collect_posts()
    log.info("Collected %d unique Bluesky posts.", len(df))
    collect_follow_edges()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/collect_reddit.py
"""LAB 2 — Reddit collector.

Primary path: PRAW in read-only OAuth (client id + secret + user agent).
Fallback:     PullPush.io (free, no key) for time-sliced historical pulls.

Key gotchas handled here:
  * comment trees are lazy  -> submission.comments.replace_more(limit=0)
  * listing 1000-item cap   -> diversify across subreddits x sort x time_filter

Run:  python -m src.collect_reddit
Outputs:  data/processed/reddit_submissions.csv , reddit_comments.csv
"""
from __future__ import annotations
from datetime import datetime, timezone
import requests
import pandas as pd

from . import config
from .utils import log, require_env, with_retries, save_csv, dedup


# ----------------------------------------------------------------------------
# PRAW (primary)
# ----------------------------------------------------------------------------
def get_reddit():
    creds = require_env("REDDIT_CLIENT_ID", "REDDIT_CLIENT_SECRET", "REDDIT_USER_AGENT")
    import praw
    reddit = praw.Reddit(
        client_id=creds["REDDIT_CLIENT_ID"],
        client_secret=creds["REDDIT_CLIENT_SECRET"],
        user_agent=creds["REDDIT_USER_AGENT"],
        check_for_async=False,
    )
    reddit.read_only = True
    log.info("Reddit client ready (read_only=%s)", reddit.read_only)
    return reddit


def _ts(utc) -> str | None:
    try:
        return datetime.fromtimestamp(float(utc), tz=timezone.utc).isoformat()
    except Exception:
        return None


def _flatten_submission(s, subreddit: str, query: str) -> dict:
    title = getattr(s, "title", "") or ""
    body = getattr(s, "selftext", "") or ""
    author = getattr(s, "author", None)
    return {
        "id": getattr(s, "id", None),
        "subreddit": subreddit,
        "matched_query": query,
        "author": str(author) if author else "[deleted]",
        "title": title,
        "selftext": body,
        "text": (title + ". " + body).strip(),
        "score": getattr(s, "score", 0),
        "upvote_ratio": getattr(s, "upvote_ratio", None),
        "num_comments": getattr(s, "num_comments", 0),
        "created_at": _ts(getattr(s, "created_utc", None)),
        "permalink": getattr(s, "permalink", None),
        "over_18": getattr(s, "over_18", None),
        "link_flair_text": getattr(s, "link_flair_text", None),
        "source": "reddit",
    }


def collect_submissions(reddit, queries=None, subreddits=None) -> pd.DataFrame:
    queries = queries or config.REDDIT_QUERIES
    subreddits = subreddits or config.SUBREDDITS
    rows: list[dict] = []
    for sub in subreddits:
        sr = reddit.subreddit(sub)
        for q in queries:
            try:
                gen = sr.search(q, sort="relevance",
                                time_filter=config.REDDIT_TIME_FILTER,
                                limit=config.REDDIT_MAX_PER_QUERY)
                n0 = len(rows)
                for s in gen:
                    rows.append(_flatten_submission(s, sub, q))
                log.info("  r/%-16s %-22s -> %d", sub, q, len(rows) - n0)
            except Exception as e:  # noqa: BLE001
                log.warning("  search failed r/%s '%s': %s", sub, q, e)
    rows = dedup(rows, key="id")
    df = pd.DataFrame(rows)
    if not df.empty:
        df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
    save_csv(df, "reddit_submissions.csv")
    return df


def collect_comments(reddit, submission_ids: list[str]) -> pd.DataFrame:
    """Flatten comment trees. replace_more(limit=0) so no replies are dropped."""
    rows: list[dict] = []
    for i, sid in enumerate(submission_ids, 1):
        try:
            sub = reddit.submission(id=sid)
            with_retries(sub.comments.replace_more, limit=0)
            for c in sub.comments.list():
                author = getattr(c, "author", None)
                rows.append({
                    "id": getattr(c, "id", None),
                    "submission_id": sid,
                    "parent_id": getattr(c, "parent_id", None),  # t1_/t3_ prefixed
                    "author": str(author) if author else "[deleted]",
                    "body": getattr(c, "body", "") or "",
                    "score": getattr(c, "score", 0),
                    "created_at": _ts(getattr(c, "created_utc", None)),
                    "subreddit": str(getattr(c, "subreddit", "")),
                    "source": "reddit",
                })
        except Exception as e:  # noqa: BLE001
            log.warning("  comments failed for %s: %s", sid, e)
        if i % 25 == 0:
            log.info("  comments: %d/%d submissions processed", i, len(submission_ids))
    df = pd.DataFrame(rows)
    if not df.empty:
        df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
    save_csv(df, "reddit_comments.csv")
    return df


# ----------------------------------------------------------------------------
# PullPush.io (fallback) — free, no key. Limits: 15/30 rpm, 1000/hour.
# ----------------------------------------------------------------------------
PULLPUSH = "https://api.pullpush.io/reddit/search"


def pullpush_search(kind: str, query: str, subreddit: str | None = None,
                    since: int | None = None, until: int | None = None,
                    size: int = config.PULLPUSH_PAGE_SIZE) -> list[dict]:
    """kind in {'submission','comment'}. since/until are epoch seconds."""
    params = {"q": query, "size": size, "sort": "desc"}
    if subreddit:
        params["subreddit"] = subreddit
    if since:
        params["since"] = since
    if until:
        params["until"] = until
    resp = with_retries(requests.get, f"{PULLPUSH}/{kind}/", params=params, timeout=60)
    resp.raise_for_status()
    return resp.json().get("data", [])


def main() -> None:
    reddit = get_reddit()
    subs = collect_submissions(reddit)
    log.info("Collected %d unique submissions.", len(subs))
    if not subs.empty:
        comments = collect_comments(reddit, subs["id"].dropna().tolist())
        log.info("Collected %d comments.", len(comments))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/corpus.py
"""Unify Bluesky posts + Reddit submissions + Reddit comments into one corpus
that the content-analysis modules (sentiment, enrichments, NER) all share.

Columns: doc_id, source, author, created_at, text, lang
"""
from __future__ import annotations
import pandas as pd

from .utils import log, load_csv, clean_basic, detect_lang


def _first_lang(langs: str) -> str:
    if isinstance(langs, str) and langs:
        return langs.split(",")[0]
    return ""


def load_documents() -> pd.DataFrame:
    frames = []

    try:
        b = load_csv("posts_bluesky.csv")
        if not b.empty:
            frames.append(pd.DataFrame({
                "doc_id": "bsky_" + b["uri"].astype(str),
                "source": "bluesky",
                "author": b["author_handle"],
                "created_at": b["created_at"],
                "text": b["text"].fillna(""),
                "lang": b["langs"].apply(_first_lang),
            }))
    except FileNotFoundError:
        pass

    try:
        s = load_csv("reddit_submissions.csv")
        if not s.empty:
            frames.append(pd.DataFrame({
                "doc_id": "rsub_" + s["id"].astype(str),
                "source": "reddit",
                "author": s["author"],
                "created_at": s["created_at"],
                "text": s["text"].fillna(""),
                "lang": "",
            }))
    except FileNotFoundError:
        pass

    try:
        c = load_csv("reddit_comments.csv")
        if not c.empty:
            frames.append(pd.DataFrame({
                "doc_id": "rcom_" + c["id"].astype(str),
                "source": "reddit",
                "author": c["author"],
                "created_at": c["created_at"],
                "text": c["body"].fillna(""),
                "lang": "",
            }))
    except FileNotFoundError:
        pass

    if not frames:
        log.warning("No collected data found; run the collectors first.")
        return pd.DataFrame(columns=["doc_id", "source", "author", "created_at", "text", "lang"])

    df = pd.concat(frames, ignore_index=True)
    df["text"] = df["text"].apply(clean_basic)
    df = df[df["text"].str.len() > 0].reset_index(drop=True)
    # fill missing language via langdetect (RQ8)
    mask = df["lang"].fillna("") == ""
    df.loc[mask, "lang"] = df.loc[mask, "text"].apply(detect_lang)
    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
    log.info("Corpus: %d documents (%s)", len(df), df["source"].value_counts().to_dict())
    return df

In [ ]:
%%writefile src/build_graph.py
"""LAB 3 — Social Network Analysis: build the directed interaction graph and
compute centrality measures.

WHY DIRECTED: "A replies to / mentions B" is asymmetric. Direction lets us
separate in-degree (attention received -> influence) from out-degree (activity),
which is what makes PageRank / directed betweenness meaningful (RQ1).
Community detection later symmetrizes to undirected (see communities.py).

Run:  python -m src.build_graph
Outputs: data/processed/graph.graphml , nodes_centrality.csv , graph_summary.txt
"""
from __future__ import annotations
import networkx as nx
import pandas as pd

from . import config
from .utils import log, load_csv, save_csv

GRAPH_PATH = config.DATA_PROCESSED / "graph.graphml"


_BAD_NODES = {"[deleted]", "None", "nan", ""}


def _add_edge(G: nx.DiGraph, u, v, relation: str, platform: str) -> None:
    if pd.isna(u) or pd.isna(v):   # NaN from empty CSV fields / missing authors
        return
    u, v = str(u).strip(), str(v).strip()
    if u == v or u in _BAD_NODES or v in _BAD_NODES:
        return
    if G.has_edge(u, v):
        G[u][v]["weight"] += 1
    else:
        G.add_edge(u, v, weight=1, relation=relation)
    for n in (u, v):
        G.nodes[n].setdefault("platform", platform)


def build_interaction_graph() -> nx.DiGraph:
    G = nx.DiGraph()

    # ---- Bluesky: mentions + replies ----
    try:
        bsky = load_csv("posts_bluesky.csv")
    except FileNotFoundError:
        bsky = pd.DataFrame()
    if not bsky.empty:
        uri2handle = dict(zip(bsky["uri"], bsky["author_handle"]))
        did2handle = dict(zip(bsky["author_did"], bsky["author_handle"]))
        for _, r in bsky.iterrows():
            src = r.get("author_handle")
            mentions = r.get("mention_dids")
            mentions = "" if pd.isna(mentions) else str(mentions)
            for did in (d.strip() for d in mentions.split(",")):
                if did:
                    _add_edge(G, src, did2handle.get(did, did), "mention", "bluesky")
            parent = r.get("reply_parent_uri")
            if isinstance(parent, str) and parent in uri2handle:
                _add_edge(G, src, uri2handle[parent], "reply", "bluesky")

    # ---- Reddit: comment author -> parent author ----
    try:
        subs = load_csv("reddit_submissions.csv")
        coms = load_csv("reddit_comments.csv")
    except FileNotFoundError:
        subs, coms = pd.DataFrame(), pd.DataFrame()
    if not coms.empty:
        sub_author = dict(zip(subs.get("id", []), subs.get("author", []))) if not subs.empty else {}
        com_author = dict(zip(coms["id"], coms["author"]))
        for _, c in coms.iterrows():
            src = c.get("author")
            pid = str(c.get("parent_id") or "")
            target = None
            if pid.startswith("t3_"):
                target = sub_author.get(pid[3:])
            elif pid.startswith("t1_"):
                target = com_author.get(pid[3:])
            _add_edge(G, src, target, "reply", "reddit")

    log.info("Interaction graph: %d nodes, %d edges", G.number_of_nodes(), G.number_of_edges())
    return G


def compute_centralities(G: nx.DiGraph) -> pd.DataFrame:
    if G.number_of_nodes() == 0:
        return pd.DataFrame()
    indeg = nx.in_degree_centrality(G)
    outdeg = nx.out_degree_centrality(G)
    # betweenness can be costly; sample on large graphs
    k = min(G.number_of_nodes(), 400) if G.number_of_nodes() > 400 else None
    btw = nx.betweenness_centrality(G, k=k, weight="weight", seed=42)
    clo = nx.closeness_centrality(G)
    pr = nx.pagerank(G, weight="weight")
    try:
        eig = nx.eigenvector_centrality_numpy(G, weight="weight")
    except Exception:
        eig = {n: float("nan") for n in G.nodes}
    rows = [{
        "node": n,
        "platform": G.nodes[n].get("platform"),
        "in_degree": G.in_degree(n),
        "out_degree": G.out_degree(n),
        "in_degree_centrality": indeg.get(n),
        "out_degree_centrality": outdeg.get(n),
        "betweenness": btw.get(n),
        "closeness": clo.get(n),
        "pagerank": pr.get(n),
        "eigenvector": eig.get(n),
    } for n in G.nodes]
    df = pd.DataFrame(rows).sort_values("pagerank", ascending=False).reset_index(drop=True)
    save_csv(df, "nodes_centrality.csv")
    return df


def graph_summary(G: nx.DiGraph) -> str:
    if G.number_of_nodes() == 0:
        return "empty graph"
    U = G.to_undirected()
    lines = [
        f"nodes: {G.number_of_nodes()}",
        f"edges: {G.number_of_edges()}",
        f"density: {nx.density(G):.5f}",
        f"transitivity (clustering): {nx.transitivity(U):.4f}",
        f"avg clustering: {nx.average_clustering(U):.4f}",
        f"reciprocity: {nx.reciprocity(G):.4f}",
        f"weakly connected components: {nx.number_weakly_connected_components(G)}",
        f"strongly connected components: {nx.number_strongly_connected_components(G)}",
        f"degree assortativity: {nx.degree_assortativity_coefficient(G):.4f}",
    ]
    txt = "\n".join(lines)
    (config.DATA_PROCESSED / "graph_summary.txt").write_text(txt)
    log.info("Graph summary:\n%s", txt)
    return txt


def save_graph(G: nx.DiGraph) -> None:
    nx.write_graphml(G, GRAPH_PATH)
    log.info("graph -> %s (open in Gephi)", GRAPH_PATH)


def load_graph() -> nx.DiGraph:
    return nx.read_graphml(GRAPH_PATH)


def main() -> None:
    G = build_interaction_graph()
    compute_centralities(G)
    graph_summary(G)
    save_graph(G)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/communities.py
"""LAB 4 — Community detection on the (symmetrized) interaction graph.

Louvain and greedy-modularity partitions are compared via modularity Q;
assortativity tests the echo-chamber hypothesis (RQ2). Community labels are
attached back to nodes so the content layer can compute per-community sentiment.

Run:  python -m src.communities  (requires src.build_graph to have run first)
Outputs: data/processed/nodes_communities.csv , communities_summary.txt
"""
from __future__ import annotations
import networkx as nx
import pandas as pd

from . import config
from .utils import log, save_csv
from .build_graph import load_graph


def to_undirected_weighted(G: nx.DiGraph) -> nx.Graph:
    """Collapse reciprocal directed edges, summing weights (for modularity)."""
    U = nx.Graph()
    U.add_nodes_from(G.nodes(data=True))
    for u, v, d in G.edges(data=True):
        w = d.get("weight", 1)
        if U.has_edge(u, v):
            U[u][v]["weight"] += w
        else:
            U.add_edge(u, v, weight=w)
    return U


def detect(U: nx.Graph) -> tuple[dict, dict, dict]:
    """Return node->community dicts for louvain & greedy, plus a metrics dict."""
    louvain = nx.community.louvain_communities(U, weight="weight", seed=42)
    greedy = nx.community.greedy_modularity_communities(U, weight="weight")

    def as_map(communities):
        return {n: i for i, com in enumerate(communities) for n in com}

    lmap, gmap = as_map(louvain), as_map(greedy)
    metrics = {
        "louvain_n_communities": len(louvain),
        "louvain_modularity": nx.community.modularity(U, louvain, weight="weight"),
        "greedy_n_communities": len(greedy),
        "greedy_modularity": nx.community.modularity(U, greedy, weight="weight"),
        "degree_assortativity": nx.degree_assortativity_coefficient(U),
    }
    return lmap, gmap, metrics


def label_top_members(U: nx.Graph, comm_map: dict, top: int = 8) -> dict[int, list[str]]:
    """Most central member handles per community (for naming the discourse camps)."""
    deg = dict(U.degree(weight="weight"))
    by_comm: dict[int, list[str]] = {}
    for node, c in comm_map.items():
        by_comm.setdefault(c, []).append(node)
    return {
        c: sorted(members, key=lambda n: deg.get(n, 0), reverse=True)[:top]
        for c, members in by_comm.items()
    }


def main() -> None:
    G = load_graph()
    U = to_undirected_weighted(G)
    if U.number_of_nodes() == 0:
        log.warning("Empty graph; run collectors + build_graph first.")
        return
    lmap, gmap, metrics = detect(U)

    df = pd.DataFrame({
        "node": list(U.nodes),
        "platform": [U.nodes[n].get("platform") for n in U.nodes],
        "community_louvain": [lmap.get(n) for n in U.nodes],
        "community_greedy": [gmap.get(n) for n in U.nodes],
        "weighted_degree": [U.degree(n, weight="weight") for n in U.nodes],
    }).sort_values(["community_louvain", "weighted_degree"], ascending=[True, False])
    save_csv(df, "nodes_communities.csv")

    tops = label_top_members(U, lmap)
    lines = [f"{k}: {v}" for k, v in metrics.items()]
    lines.append("\nTop members per Louvain community (name the camps from these):")
    for c, members in sorted(tops.items()):
        lines.append(f"  community {c} (n={sum(1 for x in lmap.values() if x == c)}): {members}")
    txt = "\n".join(lines)
    (config.DATA_PROCESSED / "communities_summary.txt").write_text(txt)
    log.info("Communities:\n%s", txt)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/content_sentiment.py
"""LAB 5 — Sentiment, emotion, and Aspect-Based Sentiment Analysis.

Lexicon lane (Lane A, word-level): VADER + AFINN + NRC emotions.
Transformer lane (Lane B, BPE): cardiffnlp twitter-roberta sentiment.
ABSA: tag each doc with product aspects (look/sound/price/...) -> aspect x sentiment.

Transformer steps degrade gracefully: if transformers/torch aren't installed,
the lexicon results are still produced.

Run:  python -m src.content_sentiment
Outputs: documents_sentiment.csv , aspect_sentiment.csv , sentiment_timeline.csv
"""
from __future__ import annotations
import pandas as pd

from . import config
from .utils import log, save_csv, clean_for_transformer
from .corpus import load_documents

EMOTIONS = ["anger", "anticipation", "disgust", "fear", "joy",
            "sadness", "surprise", "trust"]


# ---------------------------------------------------------------- lexicon lane
def add_lexicon_sentiment(df: pd.DataFrame) -> pd.DataFrame:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    from afinn import Afinn
    vader = SentimentIntensityAnalyzer()
    afinn = Afinn()

    comp = df["text"].apply(lambda t: vader.polarity_scores(t)["compound"])
    df["vader_compound"] = comp
    df["vader_label"] = pd.cut(comp, [-1.01, -0.05, 0.05, 1.01],
                               labels=["negative", "neutral", "positive"])
    df["afinn_score"] = df["text"].apply(afinn.score)
    return df


def add_emotions(df: pd.DataFrame) -> pd.DataFrame:
    try:
        from nrclex import NRCLex
    except Exception as e:  # noqa: BLE001
        log.warning("NRCLex unavailable (%s); skipping emotions.", e)
        return df
    def emo(t):
        freqs = NRCLex(t).affect_frequencies
        return [freqs.get(e, 0.0) for e in EMOTIONS]
    mat = df["text"].apply(emo).tolist()
    for i, e in enumerate(EMOTIONS):
        df[f"emo_{e}"] = [row[i] for row in mat]
    return df


# ------------------------------------------------------------ transformer lane
_PIPE_CACHE: dict = {}


def get_pipeline(task: str, model: str):
    key = (task, model)
    if key not in _PIPE_CACHE:
        from transformers import pipeline
        _PIPE_CACHE[key] = pipeline(task, model=model, truncation=True, max_length=512)
    return _PIPE_CACHE[key]


def add_transformer_sentiment(df: pd.DataFrame, batch_size: int = 32) -> pd.DataFrame:
    try:
        pipe = get_pipeline("sentiment-analysis", config.SENTIMENT_MODEL)
    except Exception as e:  # noqa: BLE001
        log.warning("transformer sentiment unavailable (%s); using VADER only.", e)
        df["transformer_label"] = df.get("vader_label")
        df["transformer_score"] = None
        return df
    texts = df["text"].apply(clean_for_transformer).tolist()
    labels, scores = [], []
    for i in range(0, len(texts), batch_size):
        for r in pipe(texts[i:i + batch_size]):
            labels.append(str(r["label"]).lower())
            scores.append(float(r["score"]))
        if i % (batch_size * 10) == 0:
            log.info("  transformer sentiment %d/%d", i, len(texts))
    df["transformer_label"] = labels
    df["transformer_score"] = scores
    return df


# --------------------------------------------------------------------- ABSA
def tag_aspects(text: str) -> list[str]:
    t = (text or "").lower()
    return [asp for asp, kws in config.ASPECTS.items() if any(k in t for k in kws)]


def aspect_table(df: pd.DataFrame, label_col: str = "transformer_label") -> pd.DataFrame:
    if label_col not in df.columns:
        label_col = "vader_label"
    df = df.copy()
    df["aspects"] = df["text"].apply(tag_aspects)
    exploded = df.explode("aspects").dropna(subset=["aspects"])
    tab = (exploded.groupby(["aspects", label_col]).size()
           .unstack(fill_value=0))
    for col in ("positive", "neutral", "negative"):
        if col not in tab.columns:
            tab[col] = 0
    tab["total"] = tab[["positive", "neutral", "negative"]].sum(axis=1)
    tab["negative_ratio"] = (tab["negative"] / tab["total"]).round(3)
    tab = tab.sort_values("total", ascending=False).reset_index()
    save_csv(tab, "aspect_sentiment.csv")
    return tab


def sentiment_timeline(df: pd.DataFrame) -> pd.DataFrame:
    d = df.dropna(subset=["created_at"]).copy()
    if d.empty:
        return pd.DataFrame()
    d = d.set_index("created_at")
    tl = d.resample("D").agg(
        n_docs=("doc_id", "count"),
        mean_compound=("vader_compound", "mean"),
    ).reset_index()
    save_csv(tl, "sentiment_timeline.csv")
    return tl


def main() -> None:
    df = load_documents()
    if df.empty:
        return
    df = add_lexicon_sentiment(df)
    df = add_emotions(df)
    df = add_transformer_sentiment(df)
    save_csv(df, "documents_sentiment.csv")
    aspect_table(df)
    sentiment_timeline(df)
    log.info("Sentiment label distribution:\n%s",
             df["transformer_label"].value_counts(dropna=False).to_string())


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/content_enrich.py
"""Content-analysis enrichments built on the Lab 5 pipeline.

  RQ6  sarcasm / irony  -> cardiffnlp/twitter-roberta-base-irony  (+ corrects sentiment)
  RQ7  stance           -> zero-shot NLI (facebook/bart-large-mnli) toward Luce & EVs
  ---  topic modelling  -> BERTopic, falling back to scikit-learn LDA
  RQ8  language          -> sentiment/stance cross-tab by language

All transformer steps degrade gracefully if the model stack is unavailable.

Run:  python -m src.content_enrich   (after content_sentiment for correction)
Outputs: documents_enriched.csv , topics_keywords.csv , language_sentiment.csv
"""
from __future__ import annotations
import pandas as pd

from . import config
from .utils import log, save_csv, load_csv, clean_for_transformer
from .corpus import load_documents
from .content_sentiment import get_pipeline

STANCE_MAX = 2000  # cap zero-shot passes for runtime sanity (logged if it bites)
TARGET_PHRASES = {"luce": "the Ferrari Luce", "ev_transition": "electric vehicles"}
STANCE_LABELS = {"in favor of": "favor", "against": "against", "neutral about": "neutral"}


def _load_base() -> pd.DataFrame:
    try:
        df = load_csv("documents_sentiment.csv")
        log.info("Loaded sentiment-scored corpus (%d docs).", len(df))
        return df
    except FileNotFoundError:
        log.warning("documents_sentiment.csv missing; using raw corpus (no correction).")
        return load_documents()


# ----------------------------------------------------------------- RQ6 sarcasm
def add_irony(df: pd.DataFrame, batch_size: int = 32) -> pd.DataFrame:
    try:
        pipe = get_pipeline("text-classification", config.IRONY_MODEL)
    except Exception as e:  # noqa: BLE001
        log.warning("irony model unavailable (%s); skipping RQ6.", e)
        df["irony_flag"] = False
        return df
    texts = df["text"].apply(clean_for_transformer).tolist()
    flags, scores = [], []
    for i in range(0, len(texts), batch_size):
        for r in pipe(texts[i:i + batch_size]):
            lab = str(r["label"]).lower()
            is_irony = ("iron" in lab and "non" not in lab) or lab in ("label_1", "1")
            flags.append(bool(is_irony))
            scores.append(float(r["score"]))
    df["irony_flag"] = flags
    df["irony_score"] = scores
    # Correction: an ironically "positive" post is really negative (RQ6).
    if "transformer_label" in df.columns:
        def correct(row):
            lab = row.get("transformer_label")
            if row["irony_flag"] and lab == "positive":
                return "negative"
            return lab
        df["sentiment_corrected"] = df.apply(correct, axis=1)
    log.info("Irony prevalence: %.1f%%", 100 * df["irony_flag"].mean())
    return df


# ------------------------------------------------------------------- RQ7 stance
def add_stance(df: pd.DataFrame) -> pd.DataFrame:
    try:
        zsc = get_pipeline("zero-shot-classification", config.STANCE_NLI_MODEL)
    except Exception as e:  # noqa: BLE001
        log.warning("stance NLI model unavailable (%s); skipping RQ7.", e)
        return df
    n = min(len(df), STANCE_MAX)
    if n < len(df):
        log.warning("stance capped at %d/%d docs for runtime.", n, len(df))
    sub = df.head(n)
    texts = sub["text"].apply(clean_for_transformer).tolist()
    label_phrases = list(STANCE_LABELS.keys())
    for target, phrase in TARGET_PHRASES.items():
        cands = [f"{lp} {phrase}" for lp in label_phrases]
        out = []
        for t in texts:
            res = zsc(t, candidate_labels=cands, multi_label=False)
            top = res["labels"][0]
            prefix = next(lp for lp in label_phrases if top.startswith(lp))
            out.append(STANCE_LABELS[prefix])
        df.loc[sub.index, f"stance_{target}"] = out
        log.info("stance(%s): %s", target,
                 pd.Series(out).value_counts().to_dict())
    return df


# --------------------------------------------------------------- topic modelling
def topic_model(df: pd.DataFrame, n_topics: int = 8) -> pd.DataFrame:
    texts = df["text"].tolist()
    # 1) BERTopic (preferred)
    try:
        from bertopic import BERTopic
        tm = BERTopic(language="multilingual", verbose=False)
        topics, _ = tm.fit_transform(texts)
        df["topic"] = topics
        info = tm.get_topic_info()[["Topic", "Count", "Name"]]
        save_csv(info.rename(columns=str.lower), "topics_keywords.csv")
        log.info("BERTopic found %d topics.", len(info))
        return df
    except Exception as e:  # noqa: BLE001
        log.warning("BERTopic unavailable (%s); falling back to sklearn LDA.", e)
    # 2) scikit-learn LDA fallback (core dep)
    try:
        from sklearn.feature_extraction.text import CountVectorizer
        from sklearn.decomposition import LatentDirichletAllocation
        vec = CountVectorizer(max_df=0.9, min_df=2, stop_words="english")
        X = vec.fit_transform(texts)
        lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
        W = lda.fit_transform(X)
        vocab = vec.get_feature_names_out()
        rows = [{"topic": k,
                 "keywords": ", ".join(vocab[i] for i in comp.argsort()[-10:][::-1])}
                for k, comp in enumerate(lda.components_)]
        save_csv(pd.DataFrame(rows), "topics_keywords.csv")
        df["topic"] = W.argmax(axis=1)
        log.info("LDA found %d topics.", n_topics)
    except Exception as e:  # noqa: BLE001
        log.warning("topic modelling failed (%s).", e)
    return df


# ------------------------------------------------------------------- RQ8 language
def language_segmentation(df: pd.DataFrame) -> pd.DataFrame:
    label_col = "sentiment_corrected" if "sentiment_corrected" in df.columns else (
        "transformer_label" if "transformer_label" in df.columns else "vader_label")
    if label_col not in df.columns:
        log.warning("no sentiment column for language segmentation.")
        return pd.DataFrame()
    d = df.copy()
    d["lang_group"] = d["lang"].where(d["lang"].isin(config.LANGUAGES), "other")
    tab = (d.groupby(["lang_group", label_col]).size().unstack(fill_value=0))
    tab["total"] = tab.sum(axis=1)
    if "negative" in tab.columns:
        tab["negative_ratio"] = (tab["negative"] / tab["total"]).round(3)
    tab = tab.reset_index()
    save_csv(tab, "language_sentiment.csv")
    log.info("Language x sentiment:\n%s", tab.to_string(index=False))
    return tab


def main() -> None:
    df = _load_base()
    if df.empty:
        return
    df = add_irony(df)
    df = add_stance(df)
    df = topic_model(df)
    save_csv(df, "documents_enriched.csv")
    language_segmentation(df)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/ner_entities.py
"""LAB 6 — Named Entity Recognition + entity-level sentiment.

spaCy (en_core_web_sm) extracts entities; we report frequency, an entity
co-occurrence network, and—joined with the sentiment layer—the sentiment
distribution *toward* each entity (e.g. "Nissan Leaf", "Jony Ive", "Vigna").

Run:  python -m src.ner_entities   (best after content_sentiment)
Outputs: entity_frequency.csv , entity_cooccurrence.csv , entity_sentiment.csv
"""
from __future__ import annotations
from collections import Counter, defaultdict
from itertools import combinations
import pandas as pd

from .utils import log, save_csv, load_csv
from .corpus import load_documents

KEEP_LABELS = {"PERSON", "ORG", "GPE", "PRODUCT", "NORP", "LOC", "FAC", "WORK_OF_ART"}


def _load_base() -> pd.DataFrame:
    try:
        return load_csv("documents_sentiment.csv")
    except FileNotFoundError:
        return load_documents()


def _nlp():
    import spacy
    from . import config
    try:
        return spacy.load(config.SPACY_MODEL, disable=["lemmatizer"])
    except OSError as e:
        raise RuntimeError(
            "spaCy model not found. Run: python -m spacy download en_core_web_sm"
        ) from e


def extract_entities(df: pd.DataFrame) -> dict[str, list[tuple[str, str]]]:
    nlp = _nlp()
    per_doc: dict[str, list[tuple[str, str]]] = {}
    texts = df["text"].fillna("").tolist()
    ids = df["doc_id"].tolist()
    for doc_id, doc in zip(ids, nlp.pipe(texts, batch_size=64)):
        ents = []
        for ent in doc.ents:
            if ent.label_ in KEEP_LABELS and len(ent.text.strip()) > 1:
                ents.append((ent.text.strip(), ent.label_))
        per_doc[doc_id] = ents
    return per_doc


def frequency_table(per_doc) -> pd.DataFrame:
    counter = Counter()
    labels = {}
    for ents in per_doc.values():
        for text, lab in ents:
            key = text.lower()
            counter[key] += 1
            labels.setdefault(key, lab)
    rows = [{"entity": k, "label": labels[k], "count": c}
            for k, c in counter.most_common()]
    df = pd.DataFrame(rows)
    save_csv(df, "entity_frequency.csv")
    return df


def cooccurrence(per_doc, min_count: int = 2) -> pd.DataFrame:
    pair = Counter()
    for ents in per_doc.values():
        uniq = sorted({t.lower() for t, _ in ents})
        for a, b in combinations(uniq, 2):
            pair[(a, b)] += 1
    rows = [{"source": a, "target": b, "weight": w}
            for (a, b), w in pair.items() if w >= min_count]
    df = pd.DataFrame(rows).sort_values("weight", ascending=False) if rows else pd.DataFrame(
        columns=["source", "target", "weight"])
    save_csv(df, "entity_cooccurrence.csv")
    return df


def entity_sentiment(df: pd.DataFrame, per_doc) -> pd.DataFrame:
    label_col = next((c for c in ("sentiment_corrected", "transformer_label", "vader_label")
                      if c in df.columns), None)
    if not label_col:
        log.warning("no sentiment column; skipping entity sentiment.")
        return pd.DataFrame()
    doc_label = dict(zip(df["doc_id"], df[label_col]))
    agg = defaultdict(Counter)
    for doc_id, ents in per_doc.items():
        lab = doc_label.get(doc_id)
        if lab is None:
            continue
        for text, _ in {(t.lower(), l) for t, l in ents}:
            agg[text][str(lab)] += 1
    rows = []
    for ent, c in agg.items():
        total = sum(c.values())
        rows.append({
            "entity": ent, "total": total,
            "positive": c.get("positive", 0),
            "neutral": c.get("neutral", 0),
            "negative": c.get("negative", 0),
            "negative_ratio": round(c.get("negative", 0) / total, 3) if total else 0,
        })
    out = pd.DataFrame(rows).sort_values("total", ascending=False)
    save_csv(out, "entity_sentiment.csv")
    return out


def main() -> None:
    df = _load_base()
    if df.empty:
        log.warning("empty corpus; run collectors first.")
        return
    per_doc = extract_entities(df)
    freq = frequency_table(per_doc)
    cooccurrence(per_doc)
    entity_sentiment(df, per_doc)
    log.info("Top entities:\n%s", freq.head(15).to_string(index=False))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/viz.py
"""Visualization helpers. Each function reads a processed CSV and writes a figure
to figures/. Functions skip silently if their input isn't there yet, so you can
run this after any subset of the pipeline.

Run:  python -m src.viz
"""
from __future__ import annotations
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

from . import config
from .utils import log, load_csv

SENT_COLORS = {"negative": "#d62728", "neutral": "#7f7f7f", "positive": "#2ca02c"}


def _save(fig, name: str):
    path = config.FIGURES / name
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    log.info("figure -> %s", path)


def _try(fn):
    try:
        fn()
    except FileNotFoundError:
        log.info("skip %s (input missing)", fn.__name__)
    except Exception as e:  # noqa: BLE001
        log.warning("%s failed: %s", fn.__name__, e)


# ------------------------------------------------------------------- content
def sentiment_distribution():
    df = load_csv("documents_enriched.csv") if (config.DATA_PROCESSED / "documents_enriched.csv").exists() else load_csv("documents_sentiment.csv")
    col = next((c for c in ("sentiment_corrected", "transformer_label", "vader_label") if c in df.columns), None)
    counts = df[col].value_counts()
    fig, ax = plt.subplots(figsize=(5, 4))
    counts.reindex(["negative", "neutral", "positive"]).plot(
        kind="bar", ax=ax, color=[SENT_COLORS.get(i, "#888") for i in ["negative", "neutral", "positive"]])
    ax.set_title(f"Sentiment distribution ({col})")
    ax.set_ylabel("documents")
    _save(fig, "sentiment_distribution.png")


def emotions():
    df = load_csv("documents_sentiment.csv")
    emo_cols = [c for c in df.columns if c.startswith("emo_")]
    if not emo_cols:
        return
    means = df[emo_cols].mean().sort_values(ascending=False)
    means.index = [c.replace("emo_", "") for c in means.index]
    fig, ax = plt.subplots(figsize=(6, 4))
    means.plot(kind="bar", ax=ax, color="#9467bd")
    ax.set_title("Mean NRC emotion intensity")
    _save(fig, "emotions.png")


def aspect_sentiment():
    tab = load_csv("aspect_sentiment.csv").set_index("aspects")
    parts = [c for c in ("negative", "neutral", "positive") if c in tab.columns]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    tab[parts].plot(kind="bar", stacked=True, ax=ax,
                    color=[SENT_COLORS[p] for p in parts])
    ax.set_title("Aspect-based sentiment (Ferrari Luce)")
    ax.set_ylabel("documents")
    _save(fig, "aspect_sentiment_stacked.png")

    if "negative_ratio" in tab.columns:
        fig2, ax2 = plt.subplots(figsize=(6, 6))
        tab["total"].plot(kind="pie", ax=ax2, autopct="%1.0f%%",
                          colors=plt.cm.Set3.colors)
        ax2.set_ylabel("")
        ax2.set_title("Share of discussion per aspect")
        _save(fig2, "aspect_share_pie.png")


def timeline():
    tl = load_csv("sentiment_timeline.csv")
    tl["created_at"] = pd.to_datetime(tl["created_at"], errors="coerce")
    fig, ax1 = plt.subplots(figsize=(9, 4))
    ax1.bar(tl["created_at"], tl["n_docs"], color="#aec7e8", label="volume")
    ax1.set_ylabel("posts / day", color="#1f77b4")
    ax2 = ax1.twinx()
    ax2.plot(tl["created_at"], tl["mean_compound"], color="#d62728", marker="o", label="mean sentiment")
    ax2.axhline(0, color="grey", lw=0.6, ls="--")
    ax2.set_ylabel("mean VADER compound", color="#d62728")
    for date, lbl in config.EVENT_DATES:
        try:
            ax1.axvline(pd.Timestamp(date, tz="UTC"), color="black", lw=0.8, ls=":")
        except Exception:
            pass
    ax1.set_title("Volume & sentiment over time")
    _save(fig, "sentiment_timeline.png")


def wordclouds():
    try:
        from wordcloud import WordCloud
    except Exception:
        return
    src = "documents_enriched.csv" if (config.DATA_PROCESSED / "documents_enriched.csv").exists() else "documents_sentiment.csv"
    df = load_csv(src)
    col = next((c for c in ("sentiment_corrected", "transformer_label", "vader_label") if c in df.columns), None)
    for sentiment in ("negative", "positive"):
        text = " ".join(df.loc[df[col] == sentiment, "text"].astype(str))
        if len(text) < 50:
            continue
        wc = WordCloud(width=800, height=400, background_color="white").generate(text)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.imshow(wc); ax.axis("off"); ax.set_title(f"{sentiment} posts")
        _save(fig, f"wordcloud_{sentiment}.png")


# ------------------------------------------------------------------- network
def centrality_top(n: int = 15):
    df = load_csv("nodes_centrality.csv").head(n)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.barh(df["node"][::-1], df["pagerank"][::-1], color="#ff7f0e")
    ax.set_title(f"Top {n} accounts by PageRank")
    _save(fig, "centrality_top.png")


def entity_frequency(n: int = 20):
    df = load_csv("entity_frequency.csv").head(n)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.barh(df["entity"][::-1], df["count"][::-1], color="#17becf")
    ax.set_title(f"Top {n} named entities")
    _save(fig, "entity_frequency.png")


def community_network():
    """Interactive hairball coloured by community (pyvis -> html)."""
    try:
        import networkx as nx
        from pyvis.network import Network
    except Exception:
        return
    G = nx.read_graphml(config.DATA_PROCESSED / "graph.graphml")
    comm = load_csv("nodes_communities.csv").set_index("node")["community_louvain"].to_dict()
    cent = load_csv("nodes_centrality.csv").set_index("node")["pagerank"].to_dict()
    net = Network(height="750px", width="100%", bgcolor="#111", font_color="white")
    palette = ["#e6194B", "#3cb44b", "#4363d8", "#f58231", "#911eb4",
               "#42d4f4", "#f032e6", "#bfef45", "#fabed4", "#469990"]
    for node in G.nodes:
        c = comm.get(node, 0) or 0
        net.add_node(node, label=str(node),
                     color=palette[int(c) % len(palette)],
                     size=8 + 600 * float(cent.get(node, 0) or 0))
    for u, v, d in G.edges(data=True):
        net.add_edge(u, v, value=d.get("weight", 1))
    out = config.FIGURES / "community_network.html"
    net.write_html(str(out))
    log.info("interactive network -> %s", out)


def main() -> None:
    for fn in (sentiment_distribution, emotions, aspect_sentiment, timeline,
               wordclouds, centrality_top, entity_frequency, community_network):
        _try(fn)


if __name__ == "__main__":
    main()

## 4. Run the pipeline

In [ ]:
import pandas as pd
from IPython.display import Image, display
import glob

### 4.1 Collect data (LAB 1–2) — Bluesky + Reddit
*Needs the credentials above.*

In [ ]:
from src.collect_bluesky import main as run_bluesky
from src.collect_reddit import main as run_reddit
run_bluesky()
run_reddit()

### 4.2 Network analysis — graph + centralities (LAB 3), communities (LAB 4)

In [ ]:
from src.build_graph import main as run_graph
from src.communities import main as run_comm
run_graph()
run_comm()
print(open("data/processed/graph_summary.txt").read())
print(open("data/processed/communities_summary.txt").read())
display(pd.read_csv("data/processed/nodes_centrality.csv").head(15))

### 4.3 Content analysis — sentiment/emotion/ABSA (LAB 5) + enrichments (sarcasm RQ6, stance RQ7, topics, language RQ8)

In [ ]:
from src.content_sentiment import main as run_sent
from src.content_enrich import main as run_enrich
run_sent()
run_enrich()
print("Aspect-based sentiment:")
display(pd.read_csv("data/processed/aspect_sentiment.csv"))
print("Sentiment by language (RQ8):")
display(pd.read_csv("data/processed/language_sentiment.csv"))

### 4.4 NER + entity-level sentiment (LAB 6)

In [ ]:
from src.ner_entities import main as run_ner
run_ner()
display(pd.read_csv("data/processed/entity_frequency.csv").head(20))
display(pd.read_csv("data/processed/entity_sentiment.csv").head(20))

### 4.5 Visualizations
Renders every chart to `figures/`. `community_network.html` is interactive — download it from the file browser.

In [ ]:
from src.viz import main as run_viz
run_viz()
for f in sorted(glob.glob("figures/*.png")):
    print(f)
    display(Image(f))

### 4.6 (Optional) Save outputs to Google Drive

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p "/content/drive/MyDrive/WSA_FerrariLuce" && cp -r data figures "/content/drive/MyDrive/WSA_FerrariLuce/"
# print("copied to Drive")